# Biochar adsorption: do machine-learning models transfer to *unseen biochars and pollutants*?

One notebook, run top to bottom (**Runtime > Run all**, about 10-20 minutes on free Colab). Internet access is needed only to download the public data set from GitHub. Nothing is uploaded or tuned by hand.

**Data:** the open compilation behind Jaffari et al., *Machine-learning-based prediction and optimization of emerging contaminants' adsorption capacity on biochar materials* (3,757 batch-adsorption measurements, 15 biochars, 14 pollutants). The notebook downloads the original `Raw_data.csv` from the authors' GitHub repository (pinned to one commit) and does not redistribute it. Please cite the original paper whenever you use these data. The repository has no licence file, so check its terms before reuse.

**What this notebook does**
1. Predicts adsorption capacity (mg/g) and removal percentage from biochar properties and experimental conditions.
2. Compares ordinary random cross-validation (the usual practice) with **held-out adsorbent-pollutant pairs**, **leave-one-biochar-out** and **leave-one-pollutant-out** validation.
3. Quantifies how many measurements are exact repeats in the inputs (replicates that random splits spread over training and test data).
4. Adds conformal prediction intervals and checks whether they cover the truth for an unseen biochar.
5. Ranks which inputs matter on held-out biochars (permutation importance).
7. **Replication:** repeats the protocol comparison and the ablation on a second public data set (heavy-metal ions on biochars, same group).
6. **New:** a replicate-free protocol (no exact input twins across train and test), an ablation that uses biochar properties only, experimental conditions only, or both, a learning curve (how many training biochars are needed), and a check of capacity derived from predicted removal.

All outputs are saved in `adsorption_results/` and zipped for download.

In [ ]:
import os, io, json, time, math, warnings, zipfile
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
import sklearn
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestRegressor, ExtraTreesRegressor, HistGradientBoostingRegressor
from sklearn.linear_model import Ridge
from sklearn.dummy import DummyRegressor
from sklearn.model_selection import GroupKFold, KFold, LeaveOneGroupOut

SEED = 42
N_BOOT = 2000
OUT = "adsorption_results"
os.makedirs(OUT, exist_ok=True)
T0 = time.time()
print("scikit-learn", sklearn.__version__, "| pandas", pd.__version__, "| numpy", np.__version__)

In [ ]:
# ---- load the public data (downloaded, not redistributed) -------------------------------
DATA_URL = ("https://raw.githubusercontent.com/ZeeshanHJ/Adsorption-capacity-prediction-for-ECs/"
            "25d8188af64b2086e2285d4e8cfef17e621c684a/Raw_data.csv")
src = os.environ.get("ADS_DATA", DATA_URL)
try:
    raw = pd.read_csv(src)
except Exception as e:
    print("Download failed:", e)
    from google.colab import files
    print("Please upload Raw_data.csv from the authors' repository.")
    up = files.upload(); raw = pd.read_csv(io.BytesIO(next(iter(up.values()))))
raw.columns = [c.strip() for c in raw.columns]
assert raw.shape == (3757, 29), raw.shape
assert raw["Adsorbent"].nunique() == 15 and raw["Pollutant"].nunique() == 14
print("rows:", len(raw), "| biochars:", raw["Adsorbent"].nunique(), "| pollutants:", raw["Pollutant"].nunique())
_d = raw["Initial concentration"] - raw["Final concentration"]; _ok = _d != 0
IDENT = float(((raw["Capacity"] * raw["Adsorbent dosage"] / _d)[_ok] - 1).abs().lt(1e-3).mean())
print(f"capacity = (Ci - Cf)/dose holds to 0.1% for {IDENT:.1%} of rows with Ci != Cf (so capacity is in mg/g if Ci is mg/L and dose g/L)")
print(raw["Pollutant"].value_counts().to_dict())

In [ ]:
# ---- cleaning, targets, features, groups --------------------------------------------------
df = raw.copy()
NUM = ["Pyrolysis temperature", "Pyrolysis time", "C", "H", "O", "N", "(O+N)/C", "Ash", "H/C", "O/C", "N/C",
       "Surface area", "Pore volume", "Average pore size", "Adsorption time", "Initial concentration", "Solution pH",
       "RPM", "Volume", "Adsorbent dosage", "Adsorption temperature", "Ion concentration", "Humic acid"]
CAT = ["Pollutant", "Wastewater type", "Adsorption type"]
# 'Final concentration' is excluded from the inputs: capacity is computed from it (it would leak the answer).
dup_share = float(df.duplicated(subset=NUM + CAT, keep=False).mean())
print(f"share of rows whose complete input vector also occurs in another row: {dup_share:.2f}")
n_neg = int((df["Capacity"] < 0).sum())
print("negative capacities (physically impossible; measurement or recording error):", n_neg, "-> excluded from the targets")
df["removal_pct"] = 100 * (df["Initial concentration"] - df["Final concentration"]) / df["Initial concentration"]
TARGETS = {"Capacity": "adsorption capacity (mg/g)", "removal_pct": "removal (%)"}
bad_rm = (df["removal_pct"] < 0) | (df["removal_pct"] > 100) | df["removal_pct"].isna()
print("rows with removal outside 0-100% or undefined:", int(bad_rm.sum()), "-> excluded from the removal target")
df.loc[df["Capacity"] < 0, "Capacity"] = np.nan
df.loc[bad_rm, "removal_pct"] = np.nan
FEATS = NUM + CAT
for c in CAT: df[c] = df[c].astype(str)
df["pair"] = df["Adsorbent"].astype(str) + "|" + df["Pollutant"].astype(str)
print("adsorbent-pollutant pairs:", df["pair"].nunique())
print(df[list(TARGETS)].describe().T[["count", "mean", "std", "min", "max"]].round(2))
print("missing share of numeric inputs:", df[NUM].isna().mean().round(2)[lambda s: s > 0].to_dict())
GROUPS = {"pair": df["pair"].values, "adsorbent": df["Adsorbent"].astype(str).values, "pollutant": df["Pollutant"].astype(str).values,
          "replicate": df[NUM + CAT].round(6).astype(str).agg("|".join, axis=1).values}
print("distinct input vectors:", len(set(GROUPS["replicate"])), "of", len(df), "rows")

In [ ]:
# ---- models and protocols -------------------------------------------------------------------
def make_pre(cat_cols, num_cols=None):
    num_cols = NUM if num_cols is None else num_cols
    num = Pipeline([("imp", SimpleImputer(strategy="median", add_indicator=True)), ("sc", StandardScaler())])
    cat = Pipeline([("imp", SimpleImputer(strategy="constant", fill_value="missing")),
                    ("oh", OneHotEncoder(handle_unknown="ignore", sparse_output=False))])
    return ColumnTransformer([("num", num, num_cols), ("cat", cat, cat_cols)])

MODELS = ["Mean", "Ridge", "RandomForest", "ExtraTrees", "HistGB"]
def make_model(name, cat_cols, n_est=200, num_cols=None):
    if name == "Mean": return DummyRegressor(strategy="mean")
    est = {"Ridge": Ridge(alpha=3.0),
           "RandomForest": RandomForestRegressor(n_estimators=n_est, max_features=0.5, min_samples_leaf=2, n_jobs=-1, random_state=SEED),
           "ExtraTrees": ExtraTreesRegressor(n_estimators=n_est, max_features=0.5, min_samples_leaf=2, n_jobs=-1, random_state=SEED),
           "HistGB": HistGradientBoostingRegressor(max_iter=150, learning_rate=0.05, max_depth=3, min_samples_leaf=10, random_state=SEED)}[name]
    return Pipeline([("pre", make_pre(cat_cols, num_cols)), ("m", est)])

PROTOCOLS = ["random", "replicate", "pair", "adsorbent", "pollutant"]
N_REP = 3
def splits_for(protocol, G, n, rep=0):
    # G: dict of group arrays (already restricted to the rows of the current target)
    if protocol == "random":    return list(KFold(5, shuffle=True, random_state=SEED + rep).split(np.arange(n)))
    if protocol == "replicate": return list(GroupKFold(5).split(np.zeros(n), groups=G["replicate"]))   # no exact input twin across train/test
    return list(LeaveOneGroupOut().split(np.zeros(n), groups=G[protocol]))

def cat_for(protocol):
    # a held-out pollutant is unknown to the model: its identity cannot be used as an input in that protocol
    return [c for c in CAT if not (protocol == "pollutant" and c == "Pollutant")]

def oof_predict(name, X, y, splits, cat_cols, keep=False):
    pred = np.full(len(y), np.nan); models = []
    for tr, te in splits:
        m = make_model(name, cat_cols); m.fit(X.iloc[tr], y[tr]); pred[te] = m.predict(X.iloc[te])
        if keep: models.append((m, te))
    assert not np.isnan(pred).any()
    return (pred, models) if keep else pred

def r2(y, p): return 1 - np.sum((y - p) ** 2) / np.sum((y - y.mean()) ** 2)

In [ ]:
# ---- main experiment ----------------------------------------------------------------------------
store = {}; rows_idx = {}; loao_models = {}
for t in TARGETS:
    ok = df[t].notna().values; rows_idx[t] = np.where(ok)[0]
    X = df.loc[ok, FEATS].reset_index(drop=True); y = df.loc[ok, t].to_numpy(float)
    G = {k: v[ok] for k, v in GROUPS.items()}
    t1 = time.time()
    for proto in PROTOCOLS:
        cc = cat_for(proto)
        sp = [splits_for(proto, G, len(y), r) for r in range(N_REP if proto == "random" else 1)]
        for m_ in MODELS:
            keep = (proto == "adsorbent" and m_ == "RandomForest")
            if keep: p0, loao_models[t] = oof_predict(m_, X, y, sp[0], cc, keep=True); prs = [p0]
            else: prs = [oof_predict(m_, X, y, s, cc) for s in sp]
            store[(t, m_, proto)] = dict(abs_err=np.mean([np.abs(y - q) for q in prs], axis=0), pred=prs[0],
                                         r2=float(np.mean([r2(y, q) for q in prs])))
    print(f"{t:12s} n={len(y)}  " + "  ".join(f"{p}: RF MAE={store[(t,'RandomForest',p)]['abs_err'].mean():.2f} (mean={store[(t,'Mean',p)]['abs_err'].mean():.2f})" for p in PROTOCOLS)
          + f"  ({time.time()-t1:.0f}s)")
print("elapsed min:", round((time.time() - T0) / 60, 1))

In [ ]:
# ---- tables with cluster bootstrap CIs (clusters = the held-out unit of each protocol) -----------
def codes_for(proto, t):
    g = GROUPS["adsorbent" if proto in ("random", "replicate") else proto][rows_idx[t]]
    codes, uniq = pd.factorize(g); return codes, len(uniq)
def boot_W(G, seed):
    return np.random.default_rng(seed).multinomial(G, np.ones(G) / G, size=N_BOOT).astype(float)
def boot_mae(a, codes, G, W):
    s = np.bincount(codes, weights=a, minlength=G); c = np.bincount(codes, minlength=G).astype(float)
    return (W @ s) / (W @ c)
def ci(x): return float(np.percentile(x, 2.5)), float(np.percentile(x, 97.5))

rows = []; gap_rows = []
for k, t in enumerate(TARGETS):
    for pi, proto in enumerate(PROTOCOLS):
        codes, G = codes_for(proto, t); W = boot_W(G, SEED + 10 * k + pi)
        bm = {m: boot_mae(store[(t, m, proto)]["abs_err"], codes, G, W) for m in MODELS}
        for m in MODELS:
            mae = store[(t, m, proto)]["abs_err"].mean(); lo, hi = ci(bm[m])
            sk = 1 - bm[m] / bm["Mean"]; sk0 = 1 - mae / store[(t, "Mean", proto)]["abs_err"].mean(); slo, shi = ci(sk)
            rows.append(dict(target=t, model=m, protocol=proto, n=len(rows_idx[t]), n_clusters=G, MAE=mae, MAE_lo=lo, MAE_hi=hi,
                             R2=store[(t, m, proto)]["r2"], skill_vs_mean=sk0, skill_lo=slo, skill_hi=shi))
        if proto != "random":
            crand = boot_mae(store[(t, "RandomForest", "random")]["abs_err"], codes, G, W)
            d = bm["RandomForest"] - crand; lo, hi = ci(d)
            gap_rows.append(dict(target=t, protocol=proto, MAE_random=store[(t, "RandomForest", "random")]["abs_err"].mean(),
                                 MAE_protocol=store[(t, "RandomForest", proto)]["abs_err"].mean(), diff=d.mean(), diff_lo=lo, diff_hi=hi))
main = pd.DataFrame(rows); gap = pd.DataFrame(gap_rows)
main.to_csv(f"{OUT}/main_results_all_models.csv", index=False); gap.to_csv(f"{OUT}/random_vs_heldout_gap.csv", index=False)
v = main[main.model == "RandomForest"].pivot(index="target", columns="protocol", values="skill_vs_mean")[PROTOCOLS]
print("Random forest skill (1 - MAE / MAE of predicting the training mean):"); print(v.round(2).to_string())
print(); print(gap.round(3).to_string(index=False))
best = main[main.protocol != "random"].pivot_table(index=["target", "protocol"], columns="model", values="MAE")[MODELS].round(3)
best["best_learned"] = best.drop(columns="Mean").idxmin(axis=1); best.to_csv(f"{OUT}/heldout_mae_by_model.csv")
print(); print(best.to_string())

In [ ]:
# ---- leakage diagnostics: exact input repeats and group sizes ------------------------------------
ok = rows_idx["Capacity"]
key = df.loc[ok, NUM + CAT].round(6).astype(str).agg("|".join, axis=1)
rep = key.map(key.value_counts())
diag = dict(rows=int(len(ok)), rows_with_identical_input_elsewhere=float((rep > 1).mean()),
            median_rows_per_input_vector=float(rep.median()), max_rows_per_input_vector=int(rep.max()),
            adsorbents=int(df.Adsorbent.nunique()), pollutants=int(df.Pollutant.nunique()), pairs=int(df.pair.nunique()))
# under random 5-fold, how often does a test row have an exact-input twin in the training folds?
sp = splits_for("random", None, len(ok), 0); tw = []
for tr, te in sp: tw.append(np.mean(key.iloc[te].isin(set(key.iloc[tr])).values))
diag["random_cv_test_rows_with_exact_input_twin_in_training"] = float(np.mean(tw))
diag["capacity_equals_dC_over_dose_share"] = IDENT
diag["distinct_input_vectors"] = int(len(set(GROUPS["replicate"])))
# difference in capacity between twins (replicates): how noisy is the label itself?
yv = df.loc[ok, "Capacity"].to_numpy(float)
gdf = pd.DataFrame({"k": key.values, "y": yv}); gdf = gdf[gdf.groupby("k")["y"].transform("size") > 1]
diag["replicate_label_MAD_mg_per_g"] = float((gdf["y"] - gdf.groupby("k")["y"].transform("mean")).abs().mean())
print(json.dumps(diag, indent=1))
sizes = pd.DataFrame({"adsorbent": df.Adsorbent.value_counts()}); sizes.to_csv(f"{OUT}/rows_per_adsorbent.csv")

In [ ]:
# ---- skill per held-out biochar and per held-out pollutant (random forest) ------------------------
by_rows = []
for t in TARGETS:
    for proto, name in (("adsorbent", "Adsorbent"), ("pollutant", "Pollutant")):
        g = GROUPS[proto][rows_idx[t]]
        for gg in sorted(set(g)):
            m_ = g == gg
            rf = store[(t, "RandomForest", proto)]["abs_err"][m_]; mn = store[(t, "Mean", proto)]["abs_err"][m_]
            by_rows.append(dict(target=t, held_out=proto, group=gg, n=int(m_.sum()), MAE_RF=rf.mean(), MAE_mean=mn.mean(), skill=1 - rf.mean() / mn.mean()))
byg = pd.DataFrame(by_rows); byg.to_csv(f"{OUT}/skill_by_heldout_group.csv", index=False)
for t in TARGETS:
    for proto in ("adsorbent", "pollutant"):
        s = byg[(byg.target == t) & (byg.held_out == proto)].skill
        print(f"{t:12s} {proto:10s} held-out groups with skill>0: {(s > 0).sum()}/{len(s)}  median {s.median():.2f}  min {s.min():.2f}  max {s.max():.2f}")

In [ ]:
# ---- conformal intervals checked on unseen biochars (random forest, Capacity) ---------------------
ALPHAS = (0.2, 0.1)
conf_rows = []; conf_g = []
for t in TARGETS:
    ok = rows_idx[t]; X = df.loc[ok, FEATS].reset_index(drop=True); y = df.loc[ok, t].to_numpy(float); g = GROUPS["adsorbent"][ok]
    pred = store[(t, "RandomForest", "adsorbent")]["pred"]
    cover = {a: np.zeros(len(y), bool) for a in ALPHAS}; width = {a: np.zeros(len(y)) for a in ALPHAS}
    for tr, te in LeaveOneGroupOut().split(np.zeros(len(y)), groups=g):
        res = []
        for itr, ite in GroupKFold(n_splits=5).split(np.zeros(len(tr)), groups=g[tr]):
            m = make_model("RandomForest", CAT, n_est=100); m.fit(X.iloc[tr[itr]], y[tr[itr]])
            res.append(np.abs(y[tr[ite]] - m.predict(X.iloc[tr[ite]])))
        res = np.concatenate(res); n = len(res)
        for a in ALPHAS:
            q = np.quantile(res, min(1.0, math.ceil((n + 1) * (1 - a)) / n), method="higher")
            cover[a][te] = np.abs(y[te] - pred[te]) <= q; width[a][te] = 2 * q
    for a in ALPHAS:
        sc = pd.DataFrame({"g": g, "c": cover[a]}).groupby("g")["c"].mean()
        conf_rows.append(dict(target=t, nominal=1 - a, coverage_pooled=cover[a].mean(), coverage_biochar_median=sc.median(),
                              coverage_biochar_min=sc.min(), share_biochars_below_nominal=(sc < 1 - a).mean(),
                              mean_width=width[a].mean(), width_over_target_sd=width[a].mean() / y.std()))
        for gg, vv in sc.items(): conf_g.append(dict(target=t, nominal=1 - a, biochar=gg, coverage=vv))
    print(t, "done", round((time.time() - T0) / 60, 1), "min")
conf = pd.DataFrame(conf_rows); conf_s = pd.DataFrame(conf_g)
conf.to_csv(f"{OUT}/conformal_coverage.csv", index=False); conf_s.to_csv(f"{OUT}/conformal_coverage_by_biochar.csv", index=False)
print(conf.round(3).to_string(index=False))

In [ ]:
# ---- permutation importance on held-out biochars (random forest) -----------------------------------
rng = np.random.default_rng(SEED); N_PERM = 3; imp_rows = []
for t in TARGETS:
    ok = rows_idx[t]; X = df.loc[ok, FEATS].reset_index(drop=True); y = df.loc[ok, t].to_numpy(float)
    base = 0.0; perm = {f: 0.0 for f in FEATS}
    for m, te in loao_models[t]:
        m.set_params(m__n_jobs=1)
        Xt = X.iloc[te].copy(); yt = y[te]; base += np.sum((yt - m.predict(Xt)) ** 2)
        for f in FEATS:
            acc = 0.0
            for _ in range(N_PERM):
                Xp = Xt.copy(); Xp[f] = rng.permutation(Xp[f].values); acc += np.sum((yt - m.predict(Xp)) ** 2)
            perm[f] += acc / N_PERM
    for f in FEATS: imp_rows.append(dict(target=t, feature=f, importance=(perm[f] - base) / base))
imp = pd.DataFrame(imp_rows); imp.to_csv(f"{OUT}/permutation_importance.csv", index=False)
for t in TARGETS: print(t, imp[imp.target == t].sort_values("importance", ascending=False).head(5)[["feature", "importance"]].round(3).values.tolist())

In [ ]:
# ---- figures ---------------------------------------------------------------------------------------
plt.rcParams.update({"font.size": 9})
rf = main[main.model == "RandomForest"]
fig, axs = plt.subplots(1, 2, figsize=(10, 3.8), sharey=True)
cols = {"random": "#4c78a8", "replicate": "#9d755d", "pair": "#f58518", "adsorbent": "#e45756", "pollutant": "#72b7b2"}
for ax, t in zip(axs, TARGETS):
    for i, pr in enumerate(PROTOCOLS):
        r = rf[(rf.target == t) & (rf.protocol == pr)].iloc[0]
        ax.bar(i, r.skill_vs_mean, color=cols[pr], yerr=[[r.skill_vs_mean - r.skill_lo], [r.skill_hi - r.skill_vs_mean]], capsize=3)
    ax.axhline(0, color="k", lw=.8); ax.set_xticks(range(5)); ax.set_xticklabels(["random\nfolds", "no input\ntwins", "held-out\npair", "held-out\nbiochar", "held-out\npollutant"], fontsize=8)
    ax.set_title(TARGETS[t])
axs[0].set_ylabel("skill vs predicting the training mean"); fig.tight_layout(); fig.savefig(f"{OUT}/fig1_skill_by_protocol.png", dpi=200); plt.close(fig)

fig, axs = plt.subplots(1, 2, figsize=(9, 4))
pols = sorted(df.Pollutant.unique()); cmap = plt.get_cmap("tab20")
for ax, t in zip(axs, TARGETS):
    y = df.loc[rows_idx[t], t].to_numpy(float); p = store[(t, "RandomForest", "adsorbent")]["pred"]; gpol = GROUPS["pollutant"][rows_idx[t]]
    for i, pl in enumerate(pols):
        m_ = gpol == pl; ax.scatter(y[m_], p[m_], s=6, alpha=.5, color=cmap(i), label=pl)
    lim = [min(y.min(), p.min()), max(y.max(), p.max())]; ax.plot(lim, lim, "k--", lw=.8)
    ax.set_xlabel("measured"); ax.set_ylabel("predicted (biochar held out)"); ax.set_title(TARGETS[t])
h_, l_ = axs[0].get_legend_handles_labels(); fig.legend(h_, l_, loc="lower center", ncol=7, fontsize=7, markerscale=2)
fig.tight_layout(rect=(0, .1, 1, 1)); fig.savefig(f"{OUT}/fig2_parity_biochar_held_out.png", dpi=200); plt.close(fig)

fig, ax = plt.subplots(figsize=(5.5, 3.8)); cs = conf_s[conf_s.target == "Capacity"]
data = [cs[np.isclose(cs.nominal, n)].coverage.values for n in (0.8, 0.9)]
ax.boxplot(data); ax.set_xticklabels(["nominal 80%", "nominal 90%"]); ax.axhline(.8, color="grey", ls="--", lw=.8); ax.axhline(.9, color="grey", ls="--", lw=.8)
ax.set_ylabel("coverage for each held-out biochar"); ax.set_title("Capacity: conformal coverage"); fig.tight_layout()
fig.savefig(f"{OUT}/fig3_conformal_coverage_by_biochar.png", dpi=200); plt.close(fig)

fig, axs = plt.subplots(1, 2, figsize=(10, 4))
for ax, t in zip(axs, TARGETS):
    d = imp[imp.target == t].sort_values("importance", ascending=False).head(8)[::-1]
    ax.barh(d.feature, d.importance, color="#4c78a8"); ax.set_title(TARGETS[t]); ax.set_xlabel("increase in squared error when shuffled")
fig.tight_layout(); fig.savefig(f"{OUT}/fig4_permutation_importance.png", dpi=200); plt.close(fig)

In [ ]:
# ---- NEW 1: what carries the signal? ablation of input groups (random forest) ---------------------------
BIO = ["Pyrolysis temperature", "Pyrolysis time", "C", "H", "O", "N", "(O+N)/C", "Ash", "H/C", "O/C", "N/C", "Surface area", "Pore volume", "Average pore size"]
COND = [c for c in NUM if c not in BIO]
SETS = {"all inputs": (NUM, CAT), "experimental conditions only": (COND, CAT), "biochar properties only": (BIO, [])}
abl_rows = []
for t in TARGETS:
    ok = rows_idx[t]; X = df.loc[ok, FEATS].reset_index(drop=True); y = df.loc[ok, t].to_numpy(float); G = {k: v[ok] for k, v in GROUPS.items()}
    for proto in ("random", "adsorbent"):
        sp = [splits_for(proto, G, len(y), r) for r in range(N_REP if proto == "random" else 1)]
        codes, Gn = codes_for(proto, t); W = boot_W(Gn, SEED + 777)
        base = boot_mae(store[(t, "Mean", proto)]["abs_err"], codes, Gn, W); mean_mae = store[(t, "Mean", proto)]["abs_err"].mean()
        for name, (nc, cc) in SETS.items():
            if name == "all inputs": errs = store[(t, "RandomForest", proto)]["abs_err"]
            else:
                prs = []
                for sset in sp:
                    pred = np.full(len(y), np.nan)
                    for tr, te in sset:
                        m = make_model("RandomForest", cc, num_cols=nc); m.fit(X.iloc[tr], y[tr]); pred[te] = m.predict(X.iloc[te])
                    prs.append(pred)
                errs = np.mean([np.abs(y - q) for q in prs], axis=0)
            lo, hi = ci(1 - boot_mae(errs, codes, Gn, W) / base)
            abl_rows.append(dict(target=t, protocol=proto, inputs=name, MAE=errs.mean(), skill=1 - errs.mean() / mean_mae, skill_lo=lo, skill_hi=hi))
    print(t, "ablation done", round((time.time() - T0) / 60, 1), "min")
abl = pd.DataFrame(abl_rows); abl.to_csv(f"{OUT}/ablation_input_groups.csv", index=False)
print(abl.round(3).to_string(index=False))

In [ ]:
# ---- NEW 2: learning curve - how many training biochars are needed? (random forest, held-out biochar) ------
K_LIST = [2, 4, 6, 8, 10, 14]; LC_REP = 5; lc_rows = []
for t in TARGETS:
    ok = rows_idx[t]; X = df.loc[ok, FEATS].reset_index(drop=True); y = df.loc[ok, t].to_numpy(float); g = GROUPS["adsorbent"][ok]
    ug = np.array(sorted(set(g))); codes, Gn = codes_for("adsorbent", t); W = boot_W(Gn, SEED + 999)
    for k in K_LIST:
        e_rf = np.zeros(len(y)); e_mn = np.zeros(len(y))
        for rep in range(LC_REP):
            rg = np.random.default_rng(SEED + 100 * k + rep)
            for gh in ug:
                te = np.where(g == gh)[0]; others = [u for u in ug if u != gh]
                tr = np.where(np.isin(g, rg.choice(others, size=k, replace=False)))[0]
                m = make_model("RandomForest", CAT, n_est=100); m.fit(X.iloc[tr], y[tr])
                e_rf[te] += np.abs(y[te] - m.predict(X.iloc[te])) / LC_REP; e_mn[te] += np.abs(y[te] - y[tr].mean()) / LC_REP
        lo, hi = ci(1 - boot_mae(e_rf, codes, Gn, W) / boot_mae(e_mn, codes, Gn, W))
        lc_rows.append(dict(target=t, k_train_biochars=k, MAE_RF=e_rf.mean(), MAE_mean=e_mn.mean(), skill=1 - e_rf.mean() / e_mn.mean(), skill_lo=lo, skill_hi=hi))
    print(t, "learning curve done", round((time.time() - T0) / 60, 1), "min")
lc = pd.DataFrame(lc_rows); lc.to_csv(f"{OUT}/learning_curve_n_biochars.csv", index=False)
print(lc.round(3).to_string(index=False))

In [ ]:
# ---- NEW 3: capacity derived from predicted removal (capacity = removal x Ci / (100 x dose)) -------------
cap_idx, rm_idx = rows_idx["Capacity"], rows_idx["removal_pct"]
common = np.intersect1d(cap_idx, rm_idx)
ic = np.searchsorted(cap_idx, common); ir = np.searchsorted(rm_idx, common)
Ci = df.loc[common, "Initial concentration"].to_numpy(float); dose = df.loc[common, "Adsorbent dosage"].to_numpy(float); ycap = df.loc[common, "Capacity"].to_numpy(float)
der_rows = []
for pi, proto in enumerate(PROTOCOLS):
    pc = store[("Capacity", "RandomForest", proto)]["pred"][ic]; pr_ = store[("removal_pct", "RandomForest", proto)]["pred"][ir]
    pm = store[("Capacity", "Mean", proto)]["pred"][ic]
    e_d = np.abs(ycap - pc); e_v = np.abs(ycap - pr_ * Ci / (100 * dose)); e_m = np.abs(ycap - pm)
    codes_c, uniq_c = pd.factorize(GROUPS["adsorbent" if proto in ("random", "replicate") else proto][common]); Gc = len(uniq_c)
    Wc = boot_W(Gc, SEED + 55 + pi)
    dlo, dhi = ci(boot_mae(e_d, codes_c, Gc, Wc) - boot_mae(e_v, codes_c, Gc, Wc))
    der_rows.append(dict(protocol=proto, n=len(common), MAE_direct=e_d.mean(), MAE_derived_from_removal=e_v.mean(), MAE_training_mean=e_m.mean(),
                         skill_direct=1 - e_d.mean() / e_m.mean(), skill_derived=1 - e_v.mean() / e_m.mean(),
                         diff_direct_minus_derived=e_d.mean() - e_v.mean(), diff_lo=dlo, diff_hi=dhi))
der = pd.DataFrame(der_rows); der.to_csv(f"{OUT}/capacity_direct_vs_derived.csv", index=False)
print(der.round(3).to_string(index=False))

In [ ]:
# ---- NEW figures: learning curve and input-group ablation ------------------------------------------------
fig, axs = plt.subplots(1, 2, figsize=(9, 3.6), sharey=True)
for ax, t in zip(axs, TARGETS):
    d = lc[lc.target == t]
    ax.errorbar(d.k_train_biochars, d.skill, yerr=[d.skill - d.skill_lo, d.skill_hi - d.skill], marker="o", capsize=3, color="#e45756")
    ax.axhline(0, color="k", lw=.8); ax.set_xlabel("number of biochars in training"); ax.set_title(TARGETS[t])
axs[0].set_ylabel("skill on the held-out biochar"); fig.tight_layout(); fig.savefig(f"{OUT}/fig5_learning_curve.png", dpi=200); plt.close(fig)

fig, axs = plt.subplots(1, 2, figsize=(9.5, 3.6), sharey=True); names = list(SETS)
for ax, t in zip(axs, TARGETS):
    for j, proto in enumerate(("random", "adsorbent")):
        for i, nm in enumerate(names):
            r = abl[(abl.target == t) & (abl.protocol == proto) & (abl.inputs == nm)].iloc[0]
            ax.bar(i + 0.38 * j - 0.19, r.skill, width=0.36, color=("#4c78a8" if proto == "random" else "#e45756"),
                   yerr=[[r.skill - r.skill_lo], [r.skill_hi - r.skill]], capsize=2, label=("random folds" if proto == "random" else "held-out biochar") if i == 0 else None)
    ax.axhline(0, color="k", lw=.8); ax.set_xticks(range(3)); ax.set_xticklabels(["all\ninputs", "conditions\nonly", "biochar\nproperties only"], fontsize=8); ax.set_title(TARGETS[t])
axs[0].set_ylabel("skill vs predicting the training mean"); axs[0].legend(fontsize=7); fig.tight_layout(); fig.savefig(f"{OUT}/fig6_input_group_ablation.png", dpi=200); plt.close(fig)

In [ ]:
# ---- REPLICATION on a second public data set: heavy-metal ions on biochars (same group) -------------------
URL2 = ("https://raw.githubusercontent.com/ZeeshanHJ/Transformer_models_prediction/086671f34cafb0d7d88a473a4eda05be32376890/HMI_data.xlsx")
src2 = os.environ.get("ADS_DATA2", URL2)
try:
    raw2 = pd.read_excel(src2)
except Exception as e:
    print("Download failed:", e)
    from google.colab import files
    print("Please upload HMI_data.xlsx from https://github.com/ZeeshanHJ/Transformer_models_prediction")
    up = files.upload(); raw2 = pd.read_excel(io.BytesIO(next(iter(up.values()))))
raw2.columns = [c.strip() for c in raw2.columns]
assert raw2.shape == (1518, 34), raw2.shape
d2 = raw2.copy()
BIO2 = ["Pyrolysis_temp", "Heating rate (oC)", "Pyrolysis_time (min)", "C", "H", "O", "N", "Ash", "H/C", "O/C", "N/C", "(O+N/C)", "Surface area", "Pore volume", "Average pore size"]
COND2 = ["radius_pm", "hydra_radius_pm", "First_ionic_IE_KJ/mol", "Adsorption_time (min)", "Ci", "solution pH", "rpm", "Volume (L)", "loading (g)", "g/L", "adsorption_temp", "Ion Concentration (M)", "DOM"]
CAT2_BIO = ["Feedstock"]; CAT2_COND = ["inorganics", "Anion_type"]
ALL_NUM2 = BIO2 + COND2; ALL_CAT2 = CAT2_BIO + CAT2_COND
d2["removal_pct"] = 100 * (d2["Ci"] - d2["Cf"]) / d2["Ci"]
n_rm_bad2 = int(((d2["removal_pct"] < 0) | (d2["removal_pct"] > 100)).sum()); n_neg2 = int((d2["qe"] < 0).sum())
d2.loc[(d2["removal_pct"] < 0) | (d2["removal_pct"] > 100), "removal_pct"] = np.nan; d2.loc[d2["qe"] < 0, "qe"] = np.nan
for c in ALL_CAT2: d2[c] = d2[c].astype(str)
d2["pair"] = d2["Adsorbent"].astype(str) + "|" + d2["inorganics"]
G2_all = {"pair": d2["pair"].values, "adsorbent": d2["Adsorbent"].astype(str).values, "pollutant": d2["inorganics"].astype(str).values,
          "replicate": d2[ALL_NUM2 + ALL_CAT2].round(6).astype(str).agg("|".join, axis=1).values}
rep2 = pd.Series(G2_all["replicate"]).map(pd.Series(G2_all["replicate"]).value_counts())
diag2 = dict(rows=int(len(d2)), adsorbents=int(d2.Adsorbent.nunique()), metals=int(d2.inorganics.nunique()), pairs=int(d2.pair.nunique()),
             rows_with_identical_input_elsewhere=float((rep2 > 1).mean()), feedstock_rows=d2.Feedstock.value_counts().to_dict(),
             negative_qe_rows=n_neg2, removal_outside_0_100_rows=n_rm_bad2)
print(json.dumps(diag2, indent=1))
T2 = {"qe": "adsorption capacity (units of the file)", "removal_pct": "removal (%)"}
def cat2_for(proto): return [c for c in ALL_CAT2 if not (proto == "pollutant" and c == "inorganics")]   # an unseen metal ion cannot be named
rep_rows = []; rep_err = {}
for k, t in enumerate(T2):
    ok = d2[t].notna().values; X = d2.loc[ok, ALL_NUM2 + ALL_CAT2].reset_index(drop=True); y = d2.loc[ok, t].to_numpy(float)
    G = {kk: v[ok] for kk, v in G2_all.items()}
    for proto in PROTOCOLS:
        cc = cat2_for(proto); sp = [splits_for(proto, G, len(y), r) for r in range(N_REP if proto == "random" else 1)]
        errs = {}
        for m_ in ("Mean", "RandomForest"):
            prs = []
            for sset in sp:
                pred = np.full(len(y), np.nan)
                for tr, te in sset:
                    m = make_model(m_, cc, num_cols=ALL_NUM2); m.fit(X.iloc[tr], y[tr]); pred[te] = m.predict(X.iloc[te])
                prs.append(pred)
            errs[m_] = np.mean([np.abs(y - q) for q in prs], axis=0)
        rep_err[(t, proto)] = errs
    for pi, proto in enumerate(PROTOCOLS):
        gcl = G["adsorbent" if proto in ("random", "replicate") else proto]; codes, uniq = pd.factorize(gcl); Gn = len(uniq)
        W = boot_W(Gn, SEED + 4000 + 10 * k + pi)
        b_rf = boot_mae(rep_err[(t, proto)]["RandomForest"], codes, Gn, W); b_mn = boot_mae(rep_err[(t, proto)]["Mean"], codes, Gn, W)
        b_r0 = boot_mae(rep_err[(t, "random")]["RandomForest"], codes, Gn, W)
        mae = rep_err[(t, proto)]["RandomForest"].mean(); mm = rep_err[(t, proto)]["Mean"].mean(); lo, hi = ci(b_rf); slo, shi = ci(1 - b_rf / b_mn); dlo, dhi = ci(b_rf - b_r0)
        rep_rows.append(dict(target=t, protocol=proto, n=len(y), n_clusters=Gn, MAE=mae, MAE_lo=lo, MAE_hi=hi, MAE_mean=mm,
                             skill=1 - mae / mm, skill_lo=slo, skill_hi=shi, diff_vs_random=mae - rep_err[(t, "random")]["RandomForest"].mean(), diff_lo=dlo, diff_hi=dhi))
    print(t, "replication protocols done", round((time.time() - T0) / 60, 1), "min")
rep_main = pd.DataFrame(rep_rows); rep_main.to_csv(f"{OUT}/replication_main.csv", index=False)
print(rep_main[["target", "protocol", "n", "n_clusters", "MAE", "skill", "skill_lo", "skill_hi", "diff_vs_random", "diff_lo", "diff_hi"]].round(2).to_string(index=False))

SETS2 = {"all inputs": (ALL_NUM2, ALL_CAT2), "experimental conditions only": (COND2, CAT2_COND), "biochar properties only": (BIO2, CAT2_BIO)}
rep_abl = []
for k, t in enumerate(T2):
    ok = d2[t].notna().values; X = d2.loc[ok, ALL_NUM2 + ALL_CAT2].reset_index(drop=True); y = d2.loc[ok, t].to_numpy(float)
    G = {kk: v[ok] for kk, v in G2_all.items()}
    for proto in ("random", "adsorbent"):
        sp = [splits_for(proto, G, len(y), r) for r in range(N_REP if proto == "random" else 1)]
        gcl = G["adsorbent"]; codes, uniq = pd.factorize(gcl); Gn = len(uniq); W = boot_W(Gn, SEED + 5000 + 10 * k + (proto == "adsorbent"))
        base = boot_mae(rep_err[(t, proto)]["Mean"], codes, Gn, W); mm = rep_err[(t, proto)]["Mean"].mean()
        for name, (nc, cc) in SETS2.items():
            if name == "all inputs": errs = rep_err[(t, proto)]["RandomForest"]
            else:
                prs = []
                for sset in sp:
                    pred = np.full(len(y), np.nan)
                    for tr, te in sset:
                        m = make_model("RandomForest", cc, num_cols=nc); m.fit(X.iloc[tr], y[tr]); pred[te] = m.predict(X.iloc[te])
                    prs.append(pred)
                errs = np.mean([np.abs(y - q) for q in prs], axis=0)
            lo, hi = ci(1 - boot_mae(errs, codes, Gn, W) / base)
            rep_abl.append(dict(target=t, protocol=proto, inputs=name, MAE=errs.mean(), skill=1 - errs.mean() / mm, skill_lo=lo, skill_hi=hi))
rep_a = pd.DataFrame(rep_abl); rep_a.to_csv(f"{OUT}/replication_ablation.csv", index=False)
print(rep_a.round(2).to_string(index=False))

In [ ]:
# ---- summary and zip --------------------------------------------------------------------------------------
summary = dict(data_url=DATA_URL, n_rows=int(len(df)), adsorbents=int(df.Adsorbent.nunique()), pollutants=int(df.Pollutant.nunique()),
               pairs=int(df.pair.nunique()), features_numeric=NUM, features_categorical=CAT, negative_capacity_rows=n_neg,
               removal_excluded_rows=int(bad_rm.sum()), diagnostics=diag, learning_curve_k=K_LIST, replication_url=URL2, replication_diagnostics=diag2, sklearn=sklearn.__version__, seed=SEED,
               bootstrap_resamples=N_BOOT, random_cv_repeats=N_REP, minutes=round((time.time() - T0) / 60, 1))
json.dump(summary, open(f"{OUT}/summary.json", "w"), indent=1)
with zipfile.ZipFile("adsorption_results.zip", "w", zipfile.ZIP_DEFLATED) as z:
    for fn in sorted(os.listdir(OUT)): z.write(f"{OUT}/{fn}", fn)
print("done in", summary["minutes"], "min; files:", sorted(os.listdir(OUT)))
try:
    from google.colab import files; files.download("adsorption_results.zip")
except Exception: pass